In [1]:
from pathlib import Path

import polars as pl
import scipy as sp

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dqdvs import dqdv_histogram

from config import DATA_PATH
ZENODO_DATA_PATH = Path(DATA_PATH) / Path("HALF_CELL_OCVS_ZENODO")

In [2]:
################################ LOAD DATA  ###################################
all_filepaths = ZENODO_DATA_PATH.rglob("*.parquet")

In [3]:
BIN_SIZE = 1e-3
cell_names = {"LNMO": 'sintef__sintef-lnmo-R2032-intelligent1-32a68c__20250424__p-ocvhold__RT.bdf.parquet', 
              "NMC 111": 'sintef__sintef-nmc111-R2032-customcells-1e88d8__20250625__p-ocvhold__RT.bdf.parquet', 
              "NMC 532": 'sintef__sintef-nmc532-R2032-gelon-f5dd84__20250607__p-ocvhold__RT.bdf.parquet', 
              "Graphite": 'sintef__sintef-graphite-R2032-intelligent-677295__20250514__p-ocvhold__RT.bdf.parquet', 
              "Silicon": 'sintef__sintef-silicon-R2032-intelligent1-882755__20250405__p-ocvhold__RT.bdf.parquet', 
              "Silicon Graphite":'sintef__sintef-silicongraphite-R2032-intelligent2-eef2ba__20250405__p-ocvhold__RT.bdf.parquet'}


# multiple material histograms

In [4]:
number_of_materials = len(cell_names)
colors = px.colors.qualitative.Plotly

fig = make_subplots(rows=number_of_materials, 
                    cols=2, 
                    shared_yaxes=True, 
                    horizontal_spacing=0.01, 
                    vertical_spacing=0.03,
                    column_widths=[0.6, 0.4],
                    #subplot_titles=[x for pair in zip(cell_names.keys(), [""]*len(cell_names)) for x in pair]
                    )

for n, (material_name, sample_name) in enumerate(cell_names.items()):

  df = pl.read_parquet(ZENODO_DATA_PATH / Path(sample_name))

  df_subset = (df
              .filter(pl.col('Step Index / 1')==2)
              .filter(pl.col('Cycle Count / 1')==3)
              )

  q = df_subset['Cumulative Capacity / Ah'].to_numpy()
  v = df_subset['Voltage / V'].to_numpy()

  q = q/q.max() #Normalized capacity
  v_dqdv, dqdv = dqdv_histogram(q=q, v=v, bin_size=BIN_SIZE)
  q_reconstructed = sp.integrate.cumulative_trapezoid(dqdv, v_dqdv, initial=0.0) + q[0]


  fig.add_trace( go.Scatter(x=q, 
                            y=v, 
                            name="OCV", 
                            line=dict(color="Black", width=4), 
                            showlegend=False), 
                          row=n+1, col=1 )
  fig.add_trace( go.Scatter(x=q_reconstructed, 
                            y=v_dqdv, 
                            name="OCV reconstructed", 
                            line=dict(color=colors[0], width=3, dash="dot"), 
                            showlegend=False), 
                          row=n+1, col=1 )
  fig.add_trace( go.Scatter(x=dqdv, 
                            y=v_dqdv, 
                            name=f"OCV histogram: {BIN_SIZE:.1e} bin size", 
                            line=dict(color=colors[0], width=2), 
                            fill='tozerox', 
                            showlegend=False), 
                          row=n+1, col=2 )
  
  fig.add_annotation(x=0.03, 
                     y=v.max()-0.1*(v.max()-v.min()),
                    text=material_name,
                    showarrow=False,
                    font=dict(size=16, weight=800), 
                    xanchor="left",
                     row=n+1, col=1 )
  
  fig.add_hline(y=v.min(),  line_color="Black", line_width=1, row=n+1, col=1)
  fig.add_hline(y=v.min(),  line_color="Black", line_width=1, row=n+1, col=2)

  fig.add_vline(x=0,  line_color="Black", line_width=1, row=n+1, col=1)
  fig.add_vline(x=0,  line_color="Black", line_width=1, row=n+1, col=2)

  fig.update_xaxes(showgrid=False, range=[0, 0.999], row=n+1, col=1)
  fig.update_xaxes(showgrid=False, row=n+1, col=2)

  fig.update_yaxes(showgrid=False, title_text="Voltage / V", row=n+1, col=1)
  fig.update_yaxes(showgrid=False, row=n+1, col=2)


fig.update_xaxes(title_text="q/Q_max", row=n+1, col=1, showgrid=False)
fig.update_xaxes(title_text="dq/dV / Ah V-1", row=n+1, col=2, showgrid=False,)


fig.update_layout(
    template="plotly_white",
    margin=dict(l=10, r=10, t=10, b=10),
    font=dict(size=18),
    width=800, 
    height=1200,
    legend=dict(
        orientation="h",    # horizontal
        yanchor="bottom",   # anchor legend's bottom
        y=1.02,             # place above the plot area
        xanchor="center",
        x=0.5
    ),)

fig.show()

In [5]:
fig.write_image("figures/histogram_several_materials.png", scale=5)